# Menjalankan Aplikasi Food & Diabetes Risk di Google Colab

Jalankan sel **berurutan dari atas ke bawah** (Shift+Enter).

Hasil akhirnya: link `https://....trycloudflare.com` yang bisa dibuka di browser/HP untuk demo.

> Catatan: link berubah setiap kali dijalankan ulang, dan aplikasi mati kalau Colab disconnect.

## 1. Upload `food-diabetes-app.zip`
Klik **Choose Files** lalu pilih file zip-nya.

In [ ]:
import os, zipfile
from google.colab import files

os.chdir('/content')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')
print('Berhasil diekstrak:', os.listdir('/content/food-diabetes-app'))

## 1b. Upload model makanan
Pilih file **`food_model_efficientnetb0.keras`** (±40 MB, tunggu sampai upload selesai 100%). File otomatis disimpan sebagai `models/food/food_model.keras`.

In [ ]:
import shutil
from google.colab import files

os.chdir('/content')
up = files.upload()
name = list(up.keys())[0]
assert name.endswith('.keras'), 'Yang di-upload harus file .keras'
os.makedirs('/content/food-diabetes-app/models/food', exist_ok=True)
shutil.move(name, '/content/food-diabetes-app/models/food/food_model.keras')
print('Model makanan siap:', os.listdir('/content/food-diabetes-app/models/food'))

## 2. Install library
TensorFlow, pandas, dll sudah ada di Colab. Cukup install Streamlit + scikit-learn versi yang sama dengan saat model diabetes disimpan (1.6.1).

In [ ]:
%cd /content/food-diabetes-app
!pip install -q streamlit scikit-learn==1.6.1
import importlib.metadata as md
print('streamlit', md.version('streamlit'), '| scikit-learn', md.version('scikit-learn'), '| tensorflow', md.version('tensorflow'))

## 3. Jalankan functional test
Harus muncul `25 passed`. Kalau ada yang gagal, jangan lanjut dulu — cek pesan errornya.

In [ ]:
!python -m pytest -q -p no:cacheprovider -W ignore

## 4. Jalankan aplikasi + buat link publik
Tunggu ±30 detik sampai link muncul.

In [ ]:
import subprocess, time, re, os

os.chdir('/content/food-diabetes-app')

# Download cloudflared (sekali saja)
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

# Matikan proses lama kalau sel ini dijalankan ulang
!pkill -f "streamlit run" ; pkill -f "cloudflared tunnel" ; true

# Jalankan Streamlit di background
st_log = open('/content/streamlit.log', 'w')
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501',
                  '--server.headless', 'true', '--browser.gatherUsageStats', 'false'],
                 stdout=st_log, stderr=st_log)
time.sleep(5)

# Buat tunnel publik
cf_log = open('/content/cloudflared.log', 'w')
subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                 stdout=cf_log, stderr=cf_log)

url = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break

print('Buka aplikasi di:', url if url else 'link belum muncul — jalankan sel 5 untuk cek log')

## 5. (Kalau ada masalah) Lihat log

In [ ]:
!tail -n 30 /content/streamlit.log
print('-----')
!tail -n 15 /content/cloudflared.log

## 6. Mengganti file dari Anggota 1 & 2 (kalau ada versi baru)
Setelah file diterima, upload ke folder yang tepat lewat sel di bawah, lalu **jalankan ulang sel 3 dan 4**.

- `food_model.keras` dan `class_names.json` → `models/food/`
- `food_nutrition.csv` → `data/` (setelah angka nutrisi diisi)

In [ ]:
import shutil
from google.colab import files

dest = {
    'food_model.keras': 'models/food/',
    'food_model_efficientnetb0.keras': 'models/food/food_model.keras',
    'class_names.json': 'models/food/',
    'food_nutrition.csv': 'data/',
}
os.chdir('/content/food-diabetes-app')
for name in files.upload():
    if name in dest:
        target = dest[name] if dest[name].endswith('.keras') else dest[name] + name
        shutil.move(name, target)
        print('OK ->', target)
    else:
        print('Nama file tidak dikenal, dilewati:', name)

## 7. (Opsional) Download project terbaru
Supaya perubahan tidak hilang saat Colab ditutup.

In [ ]:
!cd /content && rm -f food-diabetes-app-latest.zip && zip -qr food-diabetes-app-latest.zip food-diabetes-app -x '*/__pycache__/*'
files.download('/content/food-diabetes-app-latest.zip')